# compliance/01 — Decision log walkthrough

## What this demonstrates

The three main audit views Verity exposes:

- **Chronological feed.** `GET /api/v1/decisions?limit=&offset=` —   paginated list, most recent first. Shows what the system   has been doing.
- **Per-decision detail.** `GET /api/v1/decisions/{id}` — full   row including `message_history`, `tool_calls_made`,   `inference_config_snapshot`, `risk_factors`, tokens, and   duration. This is the level compliance reviewers work at.
- **Trail by pipeline run.** `GET /api/v1/audit-trail/run/{run_id}`   — every decision produced by a single pipeline invocation,   in causal order. Includes sub-agent delegations (decisions   linked to a parent via `parent_decision_id`).

Plus a simple Graphviz tree showing how sub-agent delegations nest under their parent decisions when the run included any.

**Verity capabilities exercised**

- Immutable decision log: every LLM turn and tool call is   captured in a structured row keyed by `decision_log_id`.
- Parent/child decision hierarchy (`parent_decision_id`,   `decision_depth`) that records sub-agent delegation   relationships.
- Execution-context threading: decisions can be looked up by   the business-level `execution_context_id` as well.


## Prerequisites

- Some decisions already exist (the seed data ships ~24 from   the UW demo's initial runs; `runtime/01` adds more).


In [ ]:
import os, sys
HERE = os.getcwd()
if os.path.basename(HERE) != 'ds_workbench':
    for candidate in (os.path.dirname(HERE),
                      os.path.dirname(os.path.dirname(HERE)),
                      '/home/jovyan/work'):
        if os.path.isdir(os.path.join(candidate, 'utility')):
            sys.path.insert(0, candidate); break

from utility.verity import VerityAPI, VerityAPIError
from utility.html import inject_style, badge, render_list, render_detail, render_cards

inject_style()
v = VerityAPI(application='ds_workbench')

## Execute — chronological feed

Pull the 10 most recent decisions across all applications.


In [ ]:
decisions = v.list_decisions(limit=10)
render_list(
    decisions,
    columns=[
        ('created_at','Created'),
        ('entity_type','Kind','*'),
        ('entity_name','Entity'),
        ('status','Status','*'),
        ('duration_ms','ms'),
        ('input_tokens','In tok'),
        ('output_tokens','Out tok'),
        ('application','App'),
    ],
    title=f"Latest {len(decisions)} decisions",
    description='Sorted newest first. Click through to the admin UI for richer filters.',
    empty_message='No decisions yet — run `runtime/01_run_agent.ipynb` first.',
)

## Execute — detail of one decision

Pick the first decision from the feed and fetch its full detail row, including `message_history` (Claude Messages API conversation) and `tool_calls_made` (the agentic tool turns).


In [ ]:
assert decisions, 'No decisions to drill into — run the runtime notebook first.'
first_id = decisions[0]['id']
detail = v.get_decision(first_id)

# Convert the raw message_history into a preview-sized string
# so the detail card stays readable. For full conversations,
# use decision_detail page in the admin UI.
msgs = detail.get('message_history') or []
msg_preview = '\n\n'.join(
    f"[{m.get('role','?')}] " + (
        (m['content'][:200] + ('…' if len(m['content']) > 200 else ''))
        if isinstance(m.get('content'), str) else '[structured content]'
    )
    for m in msgs[:6]
)

render_detail(
    f"Decision — {detail.get('entity_name','?')}",
    subtitle=str(detail['id']),
    header_badges=[
        (detail.get('status','?'), detail.get('status','')),
        (detail.get('entity_type','?'), detail.get('entity_type','')),
    ],
    sections=[
        {'title':'Identity','fields':[
            ('Entity',      f"{detail.get('entity_type')} / {detail.get('entity_name')}"),
            ('Version',     detail.get('version_label')),
            ('Channel',     detail.get('channel')),
            ('Application', detail.get('application')),
            ('Created',     detail.get('created_at')),
        ]},
        {'title':'Timing & usage','fields':[
            ('Duration (ms)', detail.get('duration_ms')),
            ('Input tokens',  detail.get('input_tokens')),
            ('Output tokens', detail.get('output_tokens')),
            ('Model',         detail.get('model_used')),
        ]},
        {'title':f"Tool calls made ({len(detail.get('tool_calls_made') or [])})",
         'table':{
             'columns':[
                 ('tool_name','Tool'),
                 ('transport','Transport','neutral'),
                 ('mcp_server_name','MCP server'),
                 ('status','Status','*'),
             ],
             'rows': detail.get('tool_calls_made') or [],
         }},
        {'title':f"Message history ({len(msgs)} turns)",
         'html': f'<pre style="white-space:pre-wrap;font-size:.85rem;color:#4D4D4D;'
                 f'background:#F8FAFC;padding:10px;border-radius:4px;">{msg_preview}</pre>'},
    ],
)

## Review results — trail by pipeline run

Find a decision that ran as part of a pipeline (has a `pipeline_run_id`), then pull the full trail of decisions from that pipeline invocation.


In [ ]:
# Find a recent decision that was part of a pipeline run.
pipelined = next((d for d in decisions if d.get('pipeline_run_id')), None)
if pipelined is None:
    print('No pipelined decisions found in the recent feed — skipping trail view.')
else:
    run_id = pipelined['pipeline_run_id']
    trail = v.audit_trail_by_run(run_id)
    render_list(
        trail,
        columns=[
            ('created_at','Time'),
            ('decision_depth','Depth'),
            ('entity_type','Kind','*'),
            ('entity_name','Entity'),
            ('status','Status','*'),
            ('duration_ms','ms'),
        ],
        title=f'Audit trail — pipeline_run {run_id}',
        description='Every decision in causal order, including sub-agent delegations.',
    )

In [ ]:
# Optional graphviz view of the parent→child decision tree.
# Skips gracefully when we don't have a pipelined trail above.
if pipelined is not None:
    from utility.visualizations import decision_tree
    display(decision_tree(trail, highlight_id=pipelined['id']))

---

Every view here is read-only — decision rows are immutable once written. To record a human override of an AI decision, use `POST /api/v1/overrides`; the override links to the original `decision_log_id` and is auditable separately.
